# Random Projections and the Johnson–Lindenstrauss Lemma — companion notebook

This is the **narrative** half of the `johnson-lindenstrauss` topic. One source of truth:
`johnson_lindenstrauss.py` owns every number; this notebook imports it and walks the topic
section by section, running each assert so the claims render as executed output. The full
proofs live in the MDX; here we *check* them.

Run from the repo root:
```
uv run --with numpy --with scipy --with scikit-learn --with jupyter \
    jupyter execute notebooks/johnson-lindenstrauss/01_johnson_lindenstrauss.ipynb
```

In [ ]:
import sys, pathlib
_slug, _mod = "johnson-lindenstrauss", "johnson_lindenstrauss.py"
_cands = [pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks" / _slug]
_cands += [p / "notebooks" / _slug for p in pathlib.Path.cwd().parents]
for _c in _cands:
    if (_c / _mod).exists():
        sys.path.insert(0, str(_c)); break

import numpy as np
from johnson_lindenstrauss import (
    K_GRID, structured_embeddings,
    gaussian_projection, rademacher_projection, sparse_achlioptas_projection, project,
    pairwise_sq_distortion, jl_min_dim, laurent_massart_upper,
    recall_after_projection, grid_table, finance_demo,
    test_expected_norm_preservation, test_chi_square_distribution, test_jl_lemma_holds,
    test_dimension_independence, test_projection_families_agree, test_recall_after_projection,
    test_sklearn_crosscheck, test_finance_distortion,
)
print("imported johnson_lindenstrauss")

## 1. The random map, and why it preserves norm in expectation

PCA reads the data's covariance; the Johnson–Lindenstrauss map does not look at the data at all.
Fix a target dimension $k$ and draw $A\in\mathbb R^{k\times d}$ with i.i.d. $\mathcal N(0,1)$ entries.
The map is
$$ f(x) = \tfrac{1}{\sqrt k}\,A x . $$
Because $\mathbb E[A_{ij}^2]=1$, each coordinate $(Ax)_i$ has variance $\lVert x\rVert^2$, so
$\mathbb E\lVert f(x)\rVert^2 = \lVert x\rVert^2$: **unbiased**. The Rademacher ($\pm1$) and sparse
Achlioptas families share this, since they too have unit second moment.

In [ ]:
test_expected_norm_preservation()

## 2. Concentration: the $\chi^2_k$ law and its tail

Unbiasedness alone is not enough — we need $\lVert f(x)\rVert^2$ to *concentrate*. For the Gaussian
map, $a_i\cdot x/\lVert x\rVert\sim\mathcal N(0,1)$ independently across the $k$ rows, so
$$ \frac{\lVert f(x)\rVert^2}{\lVert x\rVert^2} \sim \frac{\chi^2_k}{k}, $$
**exactly, for any ambient $d$**. The Laurent–Massart tail
$\Pr[\chi^2_k/k - 1 \ge 2\sqrt{x/k} + 2x/k]\le e^{-x}$ gives concentration at rate $e^{-ck\varepsilon^2}$.

In [ ]:
test_chi_square_distribution()

## 3. The Johnson–Lindenstrauss lemma — a union bound

To embed a whole point set, apply the concentration bound to each of the $\binom n2$ difference
vectors $u-v$ and union-bound the failures. The result: for any $\varepsilon\in(0,1)$,
$$ k \;\ge\; \frac{4\ln n}{\varepsilon^2/2 - \varepsilon^3/3} $$
suffices for a *single* random $f$ to preserve **every** pairwise squared distance to
$(1\pm\varepsilon)$. The target $k$ depends on $\ln n$ and $\varepsilon$ — **not on $d$**.

In [ ]:
test_jl_lemma_holds()

## 4. Dimension independence — and the loose constant

Two faces of the same fact. The distortion of a *fixed* vector has spread $\sqrt{2/k}$ with no $d$
in it; the *worst* pairwise distortion over a cloud grows with $n$ (more pairs to union-bound), not
with $d$. The guarantee's constant is worst-case, though: at a practical $k$ the typical pair
distorts far less than the worst — we quantify that gap in the finance demo.

In [ ]:
test_dimension_independence()

## 5. Database-friendly variants

The Gaussian matrix is dense and expensive. Achlioptas showed you can replace it with Rademacher
$\pm1$ entries, or a **sparse** matrix that is two-thirds zeros, and still get a valid JL map: each
has unit second moment (so $f$ stays unbiased) and concentrates at the same $\sqrt{2/k}$ rate.

In [ ]:
test_projection_families_agree()

## 6. Data-oblivious vs data-dependent: distances are not rankings

JL preserves *distances*. Exact nearest-neighbor *recall* is a stricter ask: on a tightly-clustered
low-rank cloud, $\pm\varepsilon$ distortion reshuffles the top-10, so random projection sacrifices
recall that data-dependent PCA — which keeps the actual signal directions — retains at the same $k$.
This is the honest contrast with the prerequisite topic.

In [ ]:
test_recall_after_projection()

## 7. Cross-check against scikit-learn

Our `jl_min_dim` matches `sklearn.random_projection.johnson_lindenstrauss_min_dim` (to within the
ceil-vs-truncate convention), and our Gaussian distortion distribution matches sklearn's
`GaussianRandomProjection`.

In [ ]:
test_sklearn_crosscheck()

## 8. The grid table and the finance case study

`grid_table()` is the block `RandomProjectionLaboratory.tsx` mirrors to the decimal; `finance_demo()`
is the $d=1536$ headline. Watch the distortion band tighten as $k$ grows, the worst-pair distortion
stay large where the typical pair is already small (the union bound), and recall split apart between
the oblivious and data-dependent projections.

In [ ]:
tbl = grid_table()
print(f"{'k':>6}{'mean':>9}{'std':>9}{'maxdev':>9}{'recall_rand':>13}{'recall_pca':>12}")
for d, r in zip(tbl['distortion'], tbl['recall']):
    print(f"{d['k']:>6}{d['mean']:>9.4f}{d['std']:>9.4f}{d['max_abs_dev']:>9.4f}"
          f"{r['rand']:>13.4f}{r['pca']:>12.4f}")
print("guaranteed k by eps:", ", ".join(f"{t['eps']}->{t['k']}" for t in tbl['threshold']))

In [ ]:
test_finance_distortion()

## What we verified

Every pedagogical claim ran as an assert: norm preservation, the $\chi^2_k$ concentration and its
Laurent–Massart tail, the union-bound JL lemma at the guaranteed $k$, dimension independence, the
database-friendly families, the oblivious-vs-dependent recall gap, the scikit-learn cross-check, and
the finance headline. The numbers above are the ones the MDX and the viz cite.

In [ ]:
print('All Johnson-Lindenstrauss claims verified against johnson_lindenstrauss.py')